In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00


# Import Libraries

In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

from utils.models import get_models
from utils.nested_cv import run_outer_fold_loop
from utils.evaluation import evaluate_model, evaluate_outer_fold

from utils.optimization import(
    get_param_dist,
    get_search_iter,
    tune_model
)

from utils.pipeline import(
    create_pipeline,
    save_fold_predictions,
    select_best_threshold
)
import shap

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load dataset and model

In [ ]:
SELECTED_MODEL = "Random Forest"

In [ ]:
ds = pd.read_csv("/content/CKD_preprocessed.csv")

X = ds.drop("d_status", axis=1)
y = ds["d_status"]

In [ ]:
models = get_models()
model_name = SELECTED_MODEL
model = models[SELECTED_MODEL]

# Create folders for result

In [ ]:
experiment4_dir = Path("/content/experiment4")

(experiment4_dir / "shap_rankings").mkdir(
    parents=True,
    exist_ok=True
)

(experiment4_dir / "reduced_models").mkdir(
    exist_ok=True
)

(experiment4_dir / "fold_results").mkdir(
    exist_ok=True
)

(experiment4_dir / "summary").mkdir(
    exist_ok=True
)

# Cross-validation

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Find the best pipeline for the selected model

In [ ]:
def get_pipeline(selected_model, estimator):
    pipeline = create_pipeline(
        selected_model,
        estimator
      )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=get_param_dist(SELECTED_MODEL),
        n_iter=get_search_iter(SELECTED_MODEL),
        scoring="roc_auc",
        cv=inner_cv,
        random_state=42,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train)

    best_pipeline = search.best_estimator_

    # print(search.best_score_)
    return best_pipeline

# Find top ranking features

In [ ]:
def shap_ranking(X_train, model):
  explainer = shap.TreeExplainer(model)
  shap_values = explainer.shap_values(X_train)

  # SHAP >= 0.45
  if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
      shap_values = shap_values[:, :, 1]

  # Older SHAP versions
  elif isinstance(shap_values, list):
      shap_values = shap_values[1]

  mean_abs_shap = np.abs(shap_values).mean(axis=0)

  importance_df = pd.DataFrame({
      "feature": X_train.columns,
      "mean_abs_shap": mean_abs_shap
  })

  importance_df = importance_df.sort_values(
      by="mean_abs_shap",
      ascending=False
  ).reset_index(drop=True)

  importance_df.insert(
      0,
      "rank",
      np.arange(1, len(importance_df) + 1)
  )
  # importance_df["rank"] = np.arange(1, len(importance_df) + 1)
  return importance_df

# Feature ranking for all folds

In [ ]:
fold_rankings = []
top10_per_fold = []
top5_per_fold = []

In [ ]:
for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y),
        start=1):

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    ####### get the best pipeline for the selected model #######
    best_pipeline = get_pipeline(SELECTED_MODEL, model)
    best_model = best_pipeline.named_steps["model"]

    ####### get top ranking features #######
    importance_df = shap_ranking(X_train, best_model)

    ####### save feature ranks per fold #######
    importance_df.to_csv(
        experiment4_dir /
        "shap_rankings" /
        f"fold_{fold}_ranking.csv",
        index=False
    )

    fold_rankings.append(importance_df)

    top10_per_fold.append(
        importance_df["feature"].head(10).tolist()
    )

    top5_per_fold.append(
        importance_df["feature"].head(5).tolist()
    )

# Combine all fold rankings

In [ ]:
all_rankings = pd.concat(
    fold_rankings,
    keys=range(1, len(fold_rankings) + 1),
    names=["fold"]
).reset_index(level=0)

all_rankings.rename(columns={"level_0": "fold"}, inplace=True)

print(all_rankings.head())

   fold  rank    feature  mean_abs_shap
0     1     1  pus_cells       0.131936
1     1     2    albumin       0.124259
2     1     3  red_cells       0.061038
3     1     4   bacteria       0.054437
4     1     5         co       0.040764


# Get a stable ranking for average SHAP

In [ ]:
stable_ranking = (
    all_rankings
    .groupby("feature")
    .agg(
        mean_shap=("mean_abs_shap", "mean"),
        std_shap=("mean_abs_shap", "std"),
        mean_rank=("rank", "mean")
    )
    .reset_index()
)

stable_ranking = stable_ranking.sort_values(
    by="mean_shap",
    ascending=False
).reset_index(drop=True)

stable_ranking.insert(
    0,
    "final_rank",
    np.arange(1, len(stable_ranking)+1)
)

stable_ranking.to_csv(
    experiment4_dir /
    "summary" /
    "stable_feature_ranking.csv",
    index=False
)

In [ ]:
stable_ranking.head()

,final_rank,feature,mean_shap,std_shap,mean_rank
0,1,pus_cells,0.134547,0.011658,1.0
1,2,albumin,0.114233,0.010401,2.0
2,3,bacteria,0.057785,0.004590,3.8
3,4,red_cells,0.055383,0.009956,3.6
4,5,co,0.051078,0.009190,4.6


# Get top 10 and top 5 features

In [ ]:
top10_features = (
    stable_ranking["feature"]
    .head(10)
    .tolist()
)

top5_features = (
    stable_ranking["feature"]
    .head(5)
    .tolist()
)

print("Top 10 Features")
print(top10_features)

print("\nTop 5 Features")
print(top5_features)

Top 10 Features
['pus_cells', 'albumin', 'bacteria', 'red_cells', 'co', 'age', 'mt', 'u_clarity', 'gc', 'ph']

Top 5 Features
['pus_cells', 'albumin', 'bacteria', 'red_cells', 'co']


In [ ]:
pd.DataFrame({
    "Top10": pd.Series(top10_features)
}).to_csv(
    experiment4_dir /
    "summary" /
    "top10_features.csv",
    index=False
)

pd.DataFrame({
    "Top5": pd.Series(top5_features)
}).to_csv(
    experiment4_dir /
    "summary" /
    "top5_features.csv",
    index=False
)

In [ ]:
X_top10 = X[top10_features].copy()

X_top5 = X[top5_features].copy()

print(X_top10.shape)
print(X_top5.shape)

(380, 10)
(380, 5)


# Verify feature stability

In [ ]:
from collections import Counter

top10_counts = Counter()

for features in top10_per_fold:
    top10_counts.update(features)

feature_frequency = (
    pd.DataFrame(
        top10_counts.items(),
        columns=["feature", "top10_frequency"]
    )
    .sort_values("top10_frequency", ascending=False)
)

feature_frequency.to_csv(
    experiment4_dir /
    "summary" /
    "top10_feature_frequency.csv",
    index=False
)

print(feature_frequency)

      feature  top10_frequency
0   pus_cells                5
1     albumin                5
2   red_cells                5
3    bacteria                5
4          co                5
5   u_clarity                5
6         age                5
9          mt                5
8          gc                4
11         ph                2
7    bpigment                1
10         kb                1
12       sp_g                1
13  epi_cells                1


# Extract Selected Model's Previous Result

In [ ]:
exp4_top10 = {}
exp4_top5 = {}
result_rows = []

In [ ]:
metric_columns = [
      "ROC AUC",
      "Accuracy",
      "PR AUC",
      "F1",
      "MCC",
      "Sensitivity",
      "Specificity",
      "Balanced Accuracy",
      "Brier Score"
  ]


In [ ]:
#### load joblib file #####
exp1_dict = joblib.load("experiment1_dataset1.joblib")
exp1_dict[SELECTED_MODEL].keys()

dict_keys(['fold_metrics', 'best_params', 'thresholds', 'best_inner_mcc', 'saved_folds', 'saved_models', 'fold_results', 'mean', 'std'])

In [ ]:
result_dict = {}
mean = exp1_dict[SELECTED_MODEL]['mean']
std = exp1_dict[SELECTED_MODEL]['std']
result_dict["Result"] = "full_feat"
for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"
result_rows.append(result_dict)
for k,v in result_dict.items():
  print(f"{k}: {v}")

Result: full_feat
ROC AUC: 0.9717 ± 0.0217
Accuracy: 0.9079 ± 0.0501
PR AUC: 0.9833 ± 0.0121
F1: 0.9295 ± 0.0354
MCC: 0.8029 ± 0.1113
Sensitivity: 0.9498 ± 0.0237
Specificity: 0.8367 ± 0.1399
Balanced Accuracy: 0.8933 ± 0.0682
Brier Score: 0.0685 ± 0.0237


# Run nested CV for top 10 features

In [ ]:
exp4_top10[SELECTED_MODEL] = {
    "fold_metrics": [],
    "best_params": [],
    "thresholds": [],
    "best_inner_mcc": [],
    "saved_folds": [],
    "saved_models": []
}

exp4_top10_dir = Path("/content/exp4_top10")

MODEL_DIR = os.path.join(exp4_top10_dir, "model_dir")
FOLD_DIR = os.path.join(exp4_top10_dir, "fold_dir")

os.makedirs(exp4_top10_dir, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FOLD_DIR, exist_ok=True)


In [ ]:
 ################# Outer fold loop #################
run_outer_fold_loop(exp4_top10, model_name, model, X_top10, y,outer_cv, inner_cv,
                    FOLD_DIR, MODEL_DIR, scoring="roc_auc")
################# Add aggregation #################
fold_df = pd.DataFrame(
        exp4_top10[model_name]["fold_metrics"]

  )
fold_df.to_csv("top10_feat_fold_results.csv", index=False, encoding='utf-8-sig')
exp4_top10[model_name]["fold_results"] = fold_df

################# Calculate mean and std for the df #################
metric_df = fold_df[metric_columns]
mean = metric_df.mean()
std = metric_df.std()

exp4_top10[model_name]["mean"] = (
        mean.to_dict()
)
exp4_top10[model_name]["std"] = (
            std.to_dict()
  )

In [ ]:
result_dict = {}
result_dict["Result"] = "top10_feat"
for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"
result_rows.append(result_dict)

for k,v in result_dict.items():
  print(f"{k}: {v}")

Result: top10_feat
ROC AUC: 0.9667 ± 0.0178
Accuracy: 0.8921 ± 0.0235
PR AUC: 0.9804 ± 0.0099
F1: 0.9169 ± 0.0155
MCC: 0.7738 ± 0.0510
Sensitivity: 0.9417 ± 0.0452
Specificity: 0.8096 ± 0.1213
Balanced Accuracy: 0.8756 ± 0.0413
Brier Score: 0.0753 ± 0.0232


# Run nested CV for top 5 features

In [ ]:
exp4_top5_dir = Path("/content/exp4_top5")

MODEL_DIR = os.path.join(exp4_top5_dir, "model_dir")
FOLD_DIR = os.path.join(exp4_top5_dir, "fold_dir")

os.makedirs(exp4_top5_dir, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FOLD_DIR, exist_ok=True)

exp4_top5[SELECTED_MODEL]  = {
    "fold_metrics": [],
    "best_params": [],
    "thresholds": [],
    "best_inner_mcc": [],
    "saved_folds": [],
    "saved_models": []
}

In [ ]:
 ################# Outer fold loop #################
run_outer_fold_loop(exp4_top5, model_name, model, X_top5, y,outer_cv, inner_cv,
                    FOLD_DIR, MODEL_DIR, scoring="roc_auc")
################# Add aggregation #################
fold_df = pd.DataFrame(
        exp4_top5[model_name]["fold_metrics"]

  )
fold_df.to_csv("top5_feat_fold_results.csv", index=False, encoding='utf-8-sig')
exp4_top5[model_name]["fold_results"] = fold_df

  ################# Calculate mean and std for the df #################
metric_df = fold_df[metric_columns]
mean = metric_df.mean()
std = metric_df.std()

exp4_top5[model_name]["mean"] = (
        mean.to_dict()
)
exp4_top5[model_name]["std"] = (
            std.to_dict()
  )


In [ ]:
result_dict = {}
result_dict["Result"] = "top5_feat"
for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"
result_rows.append(result_dict)
for k,v in result_dict.items():
  print(f"{k}: {v}")


Result: top5_feat
ROC AUC: 0.9652 ± 0.0215
Accuracy: 0.8842 ± 0.0410
PR AUC: 0.9760 ± 0.0142
F1: 0.9102 ± 0.0291
MCC: 0.7573 ± 0.0848
Sensitivity: 0.9250 ± 0.0456
Specificity: 0.8172 ± 0.1443
Balanced Accuracy: 0.8711 ± 0.0590
Brier Score: 0.0737 ± 0.0227


In [ ]:
result_df = pd.DataFrame(result_rows)
result_df.to_csv(
        "experiment4_dataset1_result.csv",
        index=False,
        encoding='utf-8-sig'
)
print(result_df)

       Result          ROC AUC         Accuracy           PR AUC  \
0   full_feat  0.9717 ± 0.0217  0.9079 ± 0.0501  0.9833 ± 0.0121   
1  top10_feat  0.9667 ± 0.0178  0.8921 ± 0.0235  0.9804 ± 0.0099   
2   top5_feat  0.9652 ± 0.0215  0.8842 ± 0.0410  0.9760 ± 0.0142   

                F1              MCC      Sensitivity      Specificity  \
0  0.9295 ± 0.0354  0.8029 ± 0.1113  0.9498 ± 0.0237  0.8367 ± 0.1399   
1  0.9169 ± 0.0155  0.7738 ± 0.0510  0.9417 ± 0.0452  0.8096 ± 0.1213   
2  0.9102 ± 0.0291  0.7573 ± 0.0848  0.9250 ± 0.0456  0.8172 ± 0.1443   

  Balanced Accuracy      Brier Score  
0   0.8933 ± 0.0682  0.0685 ± 0.0237  
1   0.8756 ± 0.0413  0.0753 ± 0.0232  
2   0.8711 ± 0.0590  0.0737 ± 0.0227  
